# LeetCode 1477 — Find Two Non-overlapping Sub-arrays Each With Target Sum bằng Python

Link đề: https://leetcode.com/problems/find-two-non-overlapping-sub-arrays-each-with-target-sum/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Dynamic Programming, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def minSumOfLengths(self, arr: list[int], target: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Tìm mọi **mảng con liên tiếp có tổng bằng `target`** bằng **prefix sum + dict** và bằng **cửa sổ trượt** (vì mọi phần tử dương).
2. Hiểu vì sao **tham lam** "lấy đoạn ngắn nhất trước" **sai**, và cần một bảng DP phụ: `best[i]` = độ dài đoạn hợp lệ **ngắn nhất kết thúc tại hoặc trước `i`**.
3. Kết hợp "đoạn hiện tại `[l..r]`" với `best[l − 1]` để đảm bảo **không chồng lấn**.
4. Biết cách tiếp cận **prefix/suffix** (gợi ý chính thức): `left[i] + right[i + 1]`.
5. Tránh lỗi: khởi tạo `best` bằng 0, ghép đoạn với chính nó, thu cửa sổ bằng `if` thay vì `while`, quên trả `-1`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và số `target`. Tìm **hai mảng con (liên tiếp) không chồng lấn**, mỗi mảng có tổng đúng bằng `target`, sao cho **tổng độ dài hai mảng nhỏ nhất**. Trả về tổng độ dài đó, hoặc `-1` nếu không tìm được hai mảng như vậy.

**Ví dụ 1:** `arr = [3,2,2,4,3], target = 3` → `2` (chỉ có `[3]` và `[3]`).

**Ví dụ 2:** `arr = [7,3,4,7], target = 7` → `2` (ba mảng `[7]`, `[3,4]`, `[7]`; chọn hai mảng `[7]`).

**Ví dụ 3:** `arr = [4,3,2,6,2,3,4], target = 6` → `-1` (chỉ có một mảng tổng 6).

**Ràng buộc:**
- `1 <= arr.length <= 10^5`
- `1 <= arr[i] <= 1000` (**mọi phần tử dương** — rất quan trọng cho cửa sổ trượt)
- `1 <= target <= 10^8`

In [1]:
import random
import time
from itertools import accumulate
from typing import List

INF = float("inf")
examples = [
    ([3, 2, 2, 4, 3], 3, 2),
    ([7, 3, 4, 7], 7, 2),
    ([4, 3, 2, 6, 2, 3, 4], 6, -1),
]
for arr, target, want in examples:
    print(f"arr={arr}, target={target} -> kỳ vọng {want}")

arr=[3, 2, 2, 4, 3], target=3 -> kỳ vọng 2
arr=[7, 3, 4, 7], target=7 -> kỳ vọng 2
arr=[4, 3, 2, 6, 2, 3, 4], target=6 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Prefix sum

`pre[i]` = tổng `i` phần tử đầu (`pre[0] = 0`). Tổng đoạn `arr[l..r]` = `pre[r+1] − pre[l]`. `itertools.accumulate` tính nhanh dãy cộng dồn.

In [2]:
arr = [7, 3, 4, 7]
pre = [0] + list(accumulate(arr))
print("pre =", pre)
l, r = 1, 2
print(f"tổng arr[{l}..{r}] =", pre[r + 1] - pre[l])

pre = [0, 7, 10, 14, 21]
tổng arr[1..2] = 7


### 3.2 Dict lưu "giá trị prefix → vị trí"

Để tìm đoạn kết thúc tại `r` có tổng `target`, cần một `l` với `pre[l] = pre[r+1] − target`. Lưu `pos[pre_value] = index` vào dict để tra `O(1)`. Vì mọi phần tử dương nên prefix **tăng ngặt** ⇒ mỗi giá trị xuất hiện đúng một lần.

In [3]:
target = 7
pos = {0: -1}               # prefix 0 "kết thúc" ở vị trí -1
s = 0
for i, x in enumerate(arr):
    s += x
    if s - target in pos:
        print(f"đoạn [{pos[s - target] + 1}..{i}] có tổng {target}")
    pos[s] = i

đoạn [0..0] có tổng 7
đoạn [1..2] có tổng 7
đoạn [3..3] có tổng 7


### 3.3 Cửa sổ trượt với phần tử dương

Hai con trỏ `l ≤ r`: tăng `r` để thêm phần tử; **trong khi** tổng `> target` thì bỏ `arr[l]` và tăng `l`. Vì các số dương, bỏ phần tử luôn làm tổng giảm ⇒ với mỗi `r` có tối đa một `l` cho tổng đúng bằng `target`.

In [4]:
l = s = 0
for r, x in enumerate(arr):
    s += x
    while s > target:
        s -= arr[l]
        l += 1
    if s == target:
        print(f"cửa sổ [{l}..{r}] tổng {s}")

cửa sổ [0..0] tổng 7
cửa sổ [1..2] tổng 7
cửa sổ [3..3] tổng 7


### 3.4 `float("inf")` làm "vô cùng"

Dùng `INF` để biểu diễn "chưa có đoạn nào": `min(INF, 3) = 3`, `INF + 2 = INF`. Cuối cùng nếu đáp án vẫn là `INF` thì trả `-1`.

In [5]:
print(min(INF, 3), INF + 2, INF > 10**18)
ans = INF
print(-1 if ans == INF else ans)

3 inf True
-1


### 3.5 Mảng DP "tốt nhất tính đến vị trí i"

`best[i] = min(best[i−1], giá trị mới tại i)` — một dạng **prefix minimum**. Nó trả lời câu hỏi "trong phần `arr[0..i]`, đoạn hợp lệ ngắn nhất dài bao nhiêu?" trong `O(1)` sau khi tính sẵn.

In [6]:
vals = [INF, 3, INF, 1, INF, 2]
best, cur = [], INF
for v in vals:
    cur = min(cur, v)
    best.append(cur)
print(best)

[inf, 3, 3, 1, 1, 1]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `arr = [7,3,4,7]`, `target = 7`. Các đoạn tổng 7: `[0..0]` (dài 1), `[1..2]` (dài 2), `[3..3]` (dài 1).

| cặp | không chồng lấn? | tổng độ dài |
|---|---|---|
| [0..0] + [1..2] | ✔ | 3 |
| [0..0] + [3..3] | ✔ | **2** |
| [1..2] + [3..3] | ✔ | 3 |

Đáp án `2`. Quan sát: nếu ta quét từ trái sang phải và mỗi khi gặp một đoạn `[l..r]` hợp lệ, ta muốn ghép nó với **đoạn ngắn nhất nằm hoàn toàn bên trái `l`** (kết thúc ≤ `l − 1`). Vậy cần biết nhanh "đoạn ngắn nhất kết thúc trước vị trí `l`" ⇒ mảng `best`.

In [7]:
def all_target_subarrays(arr, target):
    out = []
    for i in range(len(arr)):
        s = 0
        for j in range(i, len(arr)):
            s += arr[j]
            if s == target:
                out.append((i, j))
            if s > target:
                break
    return out

for arr, target, _ in examples:
    print(arr, target, "->", all_target_subarrays(arr, target))

[3, 2, 2, 4, 3] 3 -> [(0, 0), (4, 4)]
[7, 3, 4, 7] 7 -> [(0, 0), (1, 2), (3, 3)]
[4, 3, 2, 6, 2, 3, 4] 6 -> [(3, 3)]


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi đoạn, thử mọi cặp

**Ý tưởng:**
1. Liệt kê mọi đoạn `(l, r)` có tổng `target` (hai vòng lặp, dừng sớm khi tổng vượt `target`).
2. Thử mọi cặp đoạn `(a, b)` với `a.r < b.l` (không chồng lấn), lấy tổng độ dài nhỏ nhất.

In [8]:
def min_sum_brute(arr, target):
    segs = all_target_subarrays(arr, target)
    ans = INF
    for l1, r1 in segs:
        for l2, r2 in segs:
            if r1 < l2:                                  # đoạn 1 nằm trước đoạn 2
                ans = min(ans, (r1 - l1 + 1) + (r2 - l2 + 1))
    return -1 if ans == INF else ans

for arr, target, want in examples:
    got = min_sum_brute(arr, target)
    print(got, want, "✔" if got == want else "✘")

2 2 ✔
2 2 ✔
-1 -1 ✔


**Độ phức tạp:** liệt kê `O(n²)` xấu nhất (ví dụ `target` lớn); số đoạn hợp lệ `S ≤ n` (mỗi `l` có tối đa một `r` vì số dương) ⇒ thử cặp `O(S²) = O(n²)`. Với `n = 10^5`: `10^10` — quá chậm.

**Nút thắt:** với mỗi đoạn thứ hai, ta duyệt lại **mọi** đoạn thứ nhất để tìm cái ngắn nhất nằm bên trái — câu hỏi lặp lại này có thể trả lời bằng **prefix minimum**.

In [9]:
rng = random.Random(1477)
for n in (500, 1000, 2000):
    arr = [1] * n                                        # rất nhiều đoạn hợp lệ
    t = time.perf_counter()
    r = min_sum_brute(arr, 5)
    print(f"n={n}: đáp án={r}, {time.perf_counter() - t:.3f}s")

n=500: đáp án=10, 0.012s
n=1000: đáp án=10, 0.045s


n=2000: đáp án=10, 0.188s


## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam không được:** lấy đoạn ngắn nhất trước rồi tìm đoạn thứ hai có thể sai, vì đoạn ngắn nhất có thể "chắn" hai đoạn hơi dài hơn nằm hai bên (xem lỗi 1 ở mục 13).

**DP prefix-minimum:** định nghĩa `best[i]` = độ dài nhỏ nhất của một đoạn tổng `target` **kết thúc tại vị trí ≤ i** (`INF` nếu chưa có).

**Quy tắc khi quét `r` từ trái sang phải:**
- **Nếu** tồn tại đoạn hợp lệ `[l..r]` (dài `L = r − l + 1`) **thì**:
  - **nếu** `l > 0` và `best[l − 1] < INF` **thì** `ans = min(ans, L + best[l − 1])` — ghép với đoạn ngắn nhất nằm hẳn bên trái;
  - `best[r] = min(best[r − 1], L)`.
- **Ngược lại** `best[r] = best[r − 1]`.

**Invariant:** sau khi xử lý `r`, `best[r]` đúng theo định nghĩa và `ans` là đáp án tốt nhất trong số các cặp mà đoạn thứ hai kết thúc tại vị trí ≤ `r`.

Tìm `[l..r]` bằng **dict prefix** (Cách 3) hoặc **cửa sổ trượt** (Cách 4); hoặc tính `left`/`right` hai chiều (Cách 2).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix/suffix (gợi ý chính thức của đề)

**Từng đoạn:**
1. `left[i]` = đoạn hợp lệ ngắn nhất **kết thúc ≤ i** (quét trái → phải bằng cửa sổ trượt).
2. `right[i]` = đoạn hợp lệ ngắn nhất **bắt đầu ≥ i** (quét phải → trái, đối xứng).
3. Đáp án = `min(left[i] + right[i + 1])` với `i` từ `0` tới `n − 2` — điểm cắt giữa `i` và `i + 1` tách hai đoạn.

In [10]:
def shortest_ending_at_or_before(arr, target):
    n = len(arr)
    left = [INF] * n
    l = s = 0
    for r in range(n):
        s += arr[r]
        while s > target:
            s -= arr[l]; l += 1
        cur = r - l + 1 if s == target else INF
        left[r] = min(left[r - 1] if r else INF, cur)
    return left

def min_sum_prefix_suffix(arr, target):
    n = len(arr)
    left = shortest_ending_at_or_before(arr, target)
    right = shortest_ending_at_or_before(arr[::-1], target)[::-1]   # đảo mảng: "kết thúc ≤" thành "bắt đầu ≥"
    ans = min((left[i] + right[i + 1] for i in range(n - 1)), default=INF)
    return -1 if ans == INF else ans

for arr, target, want in examples:
    print(min_sum_prefix_suffix(arr, target), want)

2 2
2 2
-1 -1


Mẹo: `right` được tính bằng cách **đảo mảng**, gọi lại cùng hàm, rồi đảo kết quả. Code nộp LeetCode (Cách 2):

In [11]:
class Solution:
    def minSumOfLengths(self, arr: List[int], target: int) -> int:
        def best_prefix(a):
            res, cur, l, s = [], float("inf"), 0, 0
            for r, x in enumerate(a):
                s += x
                while s > target:
                    s -= a[l]; l += 1
                if s == target:
                    cur = min(cur, r - l + 1)
                res.append(cur)
            return res
        left = best_prefix(arr)
        right = best_prefix(arr[::-1])[::-1]
        ans = min((left[i] + right[i + 1] for i in range(len(arr) - 1)), default=float("inf"))
        return -1 if ans == float("inf") else ans

print([Solution().minSumOfLengths(a, t) for a, t, _ in examples])

[2, 2, -1]


**Độ phức tạp:** `O(n)` thời gian (mỗi con trỏ chạy tối đa `n` bước mỗi chiều), `O(n)` bộ nhớ cho `left`, `right`.

### 7.2 Cách 3 — Một lần quét: prefix sum + dict + `best`

**Từng đoạn:**
1. `pos = {0: -1}` (prefix 0 ở "vị trí −1"), `s = 0`, `best = [INF] * n`, `ans = INF`.
2. Với mỗi `r`: `s += arr[r]`. Nếu `s − target` có trong `pos` thì `l = pos[s − target] + 1` là đầu đoạn hợp lệ `[l..r]`, `L = r − l + 1`:
   - nếu `l > 0`: `ans = min(ans, L + best[l − 1])`;
   - `cur = L`, ngược lại `cur = INF`.
3. `best[r] = min(best[r − 1], cur)`; `pos[s] = r`.

Cách này **không cần** phần tử dương để tìm đoạn (dict hoạt động với mọi số nguyên nếu ta lưu vị trí gần nhất), nhưng tính đúng của "mỗi `r` một `l`" dựa trên prefix tăng ngặt.

In [12]:
def min_sum_prefix_dict(arr, target):
    n = len(arr)
    pos = {0: -1}
    best = [INF] * n
    ans = INF
    s = 0
    for r, x in enumerate(arr):
        s += x
        cur = INF
        if s - target in pos:
            l = pos[s - target] + 1
            L = r - l + 1
            if l > 0:
                ans = min(ans, L + best[l - 1])
            cur = L
        best[r] = min(best[r - 1] if r else INF, cur)
        pos[s] = r
    return -1 if ans == INF else ans

for arr, target, want in examples:
    print(min_sum_prefix_dict(arr, target), want)

2 2
2 2
-1 -1


Code nộp LeetCode (Cách 3):

In [13]:
class Solution:
    def minSumOfLengths(self, arr: List[int], target: int) -> int:
        INF = float("inf")
        pos = {0: -1}
        best = [INF] * len(arr)
        ans = INF
        s = 0
        for r, x in enumerate(arr):
            s += x
            cur = INF
            j = pos.get(s - target)
            if j is not None:                 # đoạn [j+1 .. r]
                cur = r - j
                if j >= 0:
                    ans = min(ans, cur + best[j])
            best[r] = min(best[r - 1] if r else INF, cur)
            pos[s] = r
        return -1 if ans == INF else ans

print([Solution().minSumOfLengths(a, t) for a, t, _ in examples])

[2, 2, -1]


Ở đây `j = l − 1` nên độ dài `L = r − j` và đoạn bên trái phải kết thúc ≤ `j` ⇒ dùng `best[j]`.

**Độ phức tạp:** `O(n)` thời gian trung bình (dict), `O(n)` bộ nhớ cho `pos` và `best`.

### 7.3 Cách 4 — Một lần quét: cửa sổ trượt + `best`

Giống Cách 3 nhưng tìm `[l..r]` bằng hai con trỏ (dùng được vì mọi phần tử dương) — không cần dict, hằng số nhỏ hơn.

In [14]:
def min_sum_sliding(arr, target):
    n = len(arr)
    best = [INF] * n
    ans = INF
    l = s = 0
    for r in range(n):
        s += arr[r]
        while s > target:
            s -= arr[l]
            l += 1
        cur = INF
        if s == target:
            cur = r - l + 1
            if l > 0:
                ans = min(ans, cur + best[l - 1])
        best[r] = min(best[r - 1] if r else INF, cur)
    return -1 if ans == INF else ans

for arr, target, want in examples:
    print(min_sum_sliding(arr, target), want)

2 2
2 2
-1 -1


Code nộp LeetCode (Cách 4):

In [15]:
class Solution:
    def minSumOfLengths(self, arr: List[int], target: int) -> int:
        INF = float("inf")
        n = len(arr)
        best = [INF] * n
        ans = INF
        l = s = 0
        for r in range(n):
            s += arr[r]
            while s > target:
                s -= arr[l]
                l += 1
            cur = INF
            if s == target:
                cur = r - l + 1
                if l > 0:
                    ans = min(ans, cur + best[l - 1])
            best[r] = min(best[r - 1] if r else INF, cur)
        return -1 if ans == INF else ans

print([Solution().minSumOfLengths(a, t) for a, t, _ in examples])

[2, 2, -1]


**Độ phức tạp:** `O(n)` thời gian (`l` và `r` mỗi con trỏ đi tối đa `n` bước), `O(n)` bộ nhớ cho `best`.

In [16]:
rng = random.Random(9)
big = [rng.randint(1, 1000) for _ in range(10**5)]
tg = sum(big[500:520])
for name, f in [("prefix/suffix", min_sum_prefix_suffix), ("prefix + dict", min_sum_prefix_dict),
                ("cửa sổ trượt", min_sum_sliding)]:
    t = time.perf_counter()
    r = f(big, tg)
    print(f"{name:14s}: {r}  {time.perf_counter() - t:.3f}s")

prefix/suffix : 34  0.054s
prefix + dict : 34  0.033s
cửa sổ trượt  : 34  0.020s


## 8. TRACE từng bước

Trace Cách 4 với `arr = [7,3,4,7]`, `target = 7`:

| r | arr[r] | s sau thu hẹp | l | đoạn? | cur | best[l−1] | ans | best[r] |
|---|---|---|---|---|---|---|---|---|
| 0 | 7 | 7 | 0 | [0..0] | 1 | – (l = 0) | INF | 1 |
| 1 | 3 | 3 (bỏ 7) | 1 | ✘ | INF | | INF | 1 |
| 2 | 4 | 7 | 1 | [1..2] | 2 | best[0] = 1 | 3 | 1 |
| 3 | 7 | 7 (bỏ 3, 4) | 3 | [3..3] | 1 | best[2] = 1 | **2** | 1 |

In [17]:
def trace_sliding(arr, target):
    n = len(arr)
    best = [INF] * n
    ans = INF
    l = s = 0
    for r in range(n):
        s += arr[r]
        dropped = []
        while s > target:
            s -= arr[l]; dropped.append(arr[l]); l += 1
        cur = INF
        note = ""
        if s == target:
            cur = r - l + 1
            if l > 0 and best[l - 1] < INF:
                ans = min(ans, cur + best[l - 1])
                note = f"ghép với best[{l - 1}]={best[l - 1]}"
            else:
                note = "chưa có đoạn bên trái"
        best[r] = min(best[r - 1] if r else INF, cur)
        seg = f"[{l}..{r}]" if cur < INF else "-"
        print(f"r={r} x={arr[r]} bỏ={dropped} s={s} l={l} đoạn={seg} cur={cur} {note} | ans={ans} best[r]={best[r]}")
    return -1 if ans == INF else ans

print("đáp án:", trace_sliding([7, 3, 4, 7], 7))

r=0 x=7 bỏ=[] s=7 l=0 đoạn=[0..0] cur=1 chưa có đoạn bên trái | ans=inf best[r]=1
r=1 x=3 bỏ=[7] s=3 l=1 đoạn=- cur=inf  | ans=inf best[r]=1
r=2 x=4 bỏ=[] s=7 l=1 đoạn=[1..2] cur=2 ghép với best[0]=1 | ans=3 best[r]=1
r=3 x=7 bỏ=[3, 4] s=7 l=3 đoạn=[3..3] cur=1 ghép với best[2]=1 | ans=2 best[r]=1
đáp án: 2


In [18]:
print("đáp án:", trace_sliding([1, 3, 3, 4, 3], 7))

r=0 x=1 bỏ=[] s=1 l=0 đoạn=- cur=inf  | ans=inf best[r]=inf
r=1 x=3 bỏ=[] s=4 l=0 đoạn=- cur=inf  | ans=inf best[r]=inf
r=2 x=3 bỏ=[] s=7 l=0 đoạn=[0..2] cur=3 chưa có đoạn bên trái | ans=inf best[r]=3
r=3 x=4 bỏ=[1, 3] s=7 l=2 đoạn=[2..3] cur=2 chưa có đoạn bên trái | ans=inf best[r]=2
r=4 x=3 bỏ=[3] s=7 l=3 đoạn=[3..4] cur=2 ghép với best[2]=3 | ans=5 best[r]=2
đáp án: 5


## 9. Code nộp LeetCode

Bản khuyên dùng (cửa sổ trượt + `best`, một lần quét), đúng chữ ký `minSumOfLengths(self, arr: List[int], target: int) -> int`:

In [19]:
class Solution:
    def minSumOfLengths(self, arr: List[int], target: int) -> int:
        INF = float("inf")
        n = len(arr)
        best = [INF] * n          # best[i]: đoạn tổng target ngắn nhất kết thúc tại <= i
        ans = INF
        l = s = 0
        for r in range(n):
            s += arr[r]
            while s > target:
                s -= arr[l]
                l += 1
            cur = INF
            if s == target:
                cur = r - l + 1
                if l > 0:
                    ans = min(ans, cur + best[l - 1])
            best[r] = min(best[r - 1] if r else INF, cur)
        return -1 if ans == INF else ans

for a, t, want in examples:
    assert Solution().minSumOfLengths(a, t) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Cửa sổ trượt tìm đủ đoạn.** Với mỗi `r`, sau vòng `while`, `l` là chỉ số nhỏ nhất sao cho `sum(arr[l..r]) ≤ target` (vì số dương, tổng giảm khi tăng `l`). Nếu có đoạn `[l'..r]` tổng đúng `target` thì `l' = l` (duy nhất). Do đó mọi đoạn hợp lệ đều được phát hiện tại chỉ số kết thúc của nó.
2. **`best` đúng.** Quy nạp: `best[r] = min(best[r−1], độ dài đoạn hợp lệ kết thúc tại r)` = độ dài nhỏ nhất của đoạn hợp lệ kết thúc ≤ `r`.
3. **Mọi cặp tối ưu đều được xét.** Lấy cặp tối ưu `A = [a1..a2]`, `B = [b1..b2]` với `a2 < b1`. Khi `r = b2`, thuật toán gặp `B` (`l = b1`) và xét `cur + best[b1 − 1]`. Vì `a2 ≤ b1 − 1`, `best[b1 − 1] ≤ |A|` ⇒ giá trị xét `≤ |A| + |B|`.
4. **Không bao giờ ghép đoạn chồng lấn.** Đoạn được ghép có kết thúc ≤ `l − 1` < `l` ⇒ nằm hẳn bên trái `B`. Nên mọi giá trị cập nhật vào `ans` đều là một cặp hợp lệ ⇒ `ans` không nhỏ hơn tối ưu. Kết hợp với (3) ⇒ `ans` = tối ưu.
5. **Prefix/suffix:** mọi cặp không chồng lấn có một điểm cắt `i` với `A` kết thúc ≤ `i` và `B` bắt đầu ≥ `i+1`; `left[i] + right[i+1]` là tổng nhỏ nhất qua điểm cắt đó.

## 11. Phân tích Time / Space

Gọi `n = len(arr) ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê + thử cặp | `O(n²)` (liệt kê) + `O(S²)`, `S ≤ n` | `O(S)` |
| 2. Prefix/suffix | `O(n)` | `O(n)` |
| 3. Prefix + dict + best | `O(n)` trung bình | `O(n)` |
| 4. Cửa sổ trượt + best | `O(n)` | `O(n)` |

Cửa sổ trượt: mỗi phần tử được thêm một lần (tăng `r`) và bỏ tối đa một lần (tăng `l`) ⇒ tổng `O(n)` dù có vòng `while` bên trong.

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, chỉ một đoạn, hai đoạn kề nhau, hai đoạn ngắn nhất chồng lấn, toàn số 1, `target` lớn hơn tổng), đối chiếu brute force trên mảng ngẫu nhiên (seed cố định), test hiệu năng `n = 10^5`.

In [20]:
ALL = [min_sum_brute, min_sum_prefix_suffix, min_sum_prefix_dict, min_sum_sliding,
       lambda a, t: Solution().minSumOfLengths(a, t)]

def check_all(arr, target, want=None):
    res = [f(list(arr), target) for f in ALL]
    if want is None:
        want = res[0]
    assert all(r == want for r in res), (arr, target, res)
    return want

for arr, target, want in examples:
    check_all(arr, target, want)

assert check_all([5], 5) == -1                      # chỉ một phần tử
assert check_all([5, 5], 5) == 2                    # hai đoạn kề nhau
assert check_all([1, 2, 1], 3) == -1                # hai đoạn [1,2], [2,1] chồng lấn
assert check_all([1, 3, 3, 4, 3], 7) == 5           # đoạn ngắn nhất [3,4] chắn hai đoạn khác
assert check_all([1] * 10, 3) == 6
assert check_all([1, 1, 1], 100) == -1              # target lớn hơn tổng
assert check_all([2, 1, 3, 3, 2, 3, 1], 6) == 5     # nhiều lựa chọn
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [21]:
rng = random.Random(2021)
cnt = {"-1": 0, "có": 0}
for _ in range(3000):
    n = rng.randint(1, 25)
    hi = rng.choice([3, 10, 1000])
    arr = [rng.randint(1, hi) for _ in range(n)]
    if rng.random() < 0.5 and n >= 2:
        i, j = sorted(rng.sample(range(n + 1), 2))
        target = sum(arr[i:j])                      # bảo đảm có ít nhất một đoạn
    else:
        target = rng.randint(1, 3 * hi)
    r = check_all(arr, target)
    cnt["-1" if r == -1 else "có"] += 1
print("Random 3000 test: OK", cnt)

Random 3000 test: OK {'-1': 1936, 'có': 1064}


In [22]:
# Ràng buộc lớn
rng = random.Random(4)
big = [rng.randint(1, 1000) for _ in range(10**5)]
for tg in (sum(big[10:15]), 10**8, 1000):
    t = time.perf_counter()
    r = Solution().minSumOfLengths(big, tg)
    dt = time.perf_counter() - t
    assert r == min_sum_prefix_dict(big, tg)
    print(f"target={tg}: {r}, {dt:.3f}s")
t = time.perf_counter()
print("toàn số 1, target=2:", Solution().minSumOfLengths([1] * 10**5, 2), f"{time.perf_counter() - t:.3f}s")

target=3032: 8, 0.020s
target=100000000: -1, 0.013s
target=1000: 2, 0.019s


toàn số 1, target=2: 4 0.022s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: lấy đoạn ngắn nhất trước, rồi tìm đoạn thứ hai

In [23]:
def wrong_greedy(arr, target):
    segs = sorted(all_target_subarrays(arr, target), key=lambda p: (p[1] - p[0], p[0]))
    if not segs:
        return -1
    a = segs[0]                                       # SAI: cố định đoạn ngắn nhất
    for b in segs[1:]:
        if b[1] < a[0] or a[1] < b[0]:
            return (a[1] - a[0] + 1) + (b[1] - b[0] + 1)
    return -1

arr, target = [1, 3, 3, 4, 3], 7
print("các đoạn:", all_target_subarrays(arr, target))
print("SAI :", wrong_greedy(arr, target), "| đúng:", min_sum_sliding(arr, target))

các đoạn: [(0, 2), (2, 3), (3, 4)]
SAI : -1 | đúng: 5


Đoạn ngắn nhất `[3,4]` (chỉ số 2..3) chồng lấn với cả `[1,3,3]` và `[4,3]`, nhưng hai đoạn này lại không chồng lấn nhau (`3 + 2 = 5`). **Sửa:** DP `best` xét mọi đoạn thứ hai.

### Lỗi 2 — Lấy giá trị "trước vị trí 0" của `best` bằng `0`

Khi chưa có đoạn bên trái, nếu `best` mặc định là `0` thì một đoạn đơn lẻ được tính như "một cặp" (ghép với đoạn dài 0 không tồn tại).

In [24]:
def wrong_best_zero(arr, target):
    n = len(arr)
    best = [0] * n
    ans, l, s = INF, 0, 0
    for r in range(n):
        s += arr[r]
        while s > target:
            s -= arr[l]; l += 1
        cur = INF
        if s == target:
            cur = r - l + 1
            if l > 0:
                ans = min(ans, cur + best[l - 1])
        best[r] = min(best[r - 1] if r else 0, cur)   # SAI: "chưa có" phải là INF, không phải 0
    return -1 if ans == INF else ans

arr, target, want = examples[2]
print("SAI :", wrong_best_zero(arr, target), "| đúng:", want)

SAI : 1 | đúng: -1


**Sửa:** `best = [INF] * n` và `best[-1]` "ảo" là `INF`.

### Lỗi 3 — Ghép đoạn với chính nó (dùng `best[r]` thay vì `best[l − 1]`)

In [25]:
def wrong_self_pair(arr, target):
    n = len(arr)
    best = [INF] * n
    ans, l, s = INF, 0, 0
    for r in range(n):
        s += arr[r]
        while s > target:
            s -= arr[l]; l += 1
        cur = r - l + 1 if s == target else INF
        best[r] = min(best[r - 1] if r else INF, cur)
        if cur < INF:
            ans = min(ans, cur + best[r])             # SAI: best[r] có thể chính là đoạn hiện tại
    return -1 if ans == INF else ans

for arr, target in ([3], 3), ([1, 2, 1], 3):
    print("SAI :", wrong_self_pair(arr, target), "| đúng:", min_sum_sliding(arr, target), "| arr =", arr)

SAI : 2 | đúng: -1 | arr = [3]
SAI : 4 | đúng: -1 | arr = [1, 2, 1]


**Sửa:** chỉ ghép với `best[l − 1]` (đoạn kết thúc **trước** điểm bắt đầu của đoạn hiện tại).

### Lỗi 4 — Thu hẹp cửa sổ bằng `if` thay vì `while`

In [26]:
def wrong_if_shrink(arr, target):
    n = len(arr)
    best = [INF] * n
    ans, l, s = INF, 0, 0
    for r in range(n):
        s += arr[r]
        if s > target:                                # SAI: chỉ bỏ được 1 phần tử
            s -= arr[l]; l += 1
        cur = INF
        if s == target:
            cur = r - l + 1
            if l > 0:
                ans = min(ans, cur + best[l - 1])
        best[r] = min(best[r - 1] if r else INF, cur)
    return -1 if ans == INF else ans

arr, target = [1, 1, 1, 3], 3
print("SAI :", wrong_if_shrink(arr, target), "| đúng:", min_sum_sliding(arr, target))

SAI : -1 | đúng: 4


Khi thêm `3`, tổng thành `6`; bỏ một số `1` còn `5 > 3` nên bỏ lỡ đoạn `[3]`. **Sửa:** `while s > target`.

### Lỗi 5 — Quên đổi `INF` thành `-1`

In [27]:
def wrong_return_inf(arr, target):
    n = len(arr)
    best = [INF] * n
    ans, l, s = INF, 0, 0
    for r in range(n):
        s += arr[r]
        while s > target:
            s -= arr[l]; l += 1
        cur = INF
        if s == target:
            cur = r - l + 1
            if l > 0:
                ans = min(ans, cur + best[l - 1])
        best[r] = min(best[r - 1] if r else INF, cur)
    return ans                                        # SAI: quên đổi INF thành -1

arr, target, want = examples[2]
print("SAI :", wrong_return_inf(arr, target), "| đúng:", want)

SAI : inf | đúng: -1


**Sửa:** `return -1 if ans == INF else ans`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai đoạn không chồng lấn, tối ưu tổng" ⇒ **prefix best + quét** (hoặc prefix/suffix với điểm cắt). Cùng họ với 123 Best Time to Buy and Sell Stock III, 689 Maximum Sum of 3 Non-Overlapping Subarrays.
- "Đoạn con tổng bằng `k`" + **phần tử dương** ⇒ **cửa sổ trượt**; có số âm ⇒ **prefix sum + dict**.
- Đừng tin tham lam khi lựa chọn đầu tiên **chặn** lựa chọn sau — luôn thử phản ví dụ nhỏ bằng brute force.
- `INF` cho "chưa có", nhớ đổi về `-1` khi trả.
- Ghép "đoạn hiện tại" với "tốt nhất **trước** điểm bắt đầu" — chỉ số `l − 1`, không phải `r`.

## 15. Biến thể của bài

1. **Có số âm hoặc 0:** cửa sổ trượt sai; dùng prefix sum + dict (lưu vị trí **gần nhất** của mỗi prefix để có đoạn ngắn nhất kết thúc tại `r`).
2. **Ba đoạn không chồng lấn** (hoặc `k` đoạn): DP `dp[j][i]` = tổng nhỏ nhất dùng `j` đoạn trong `arr[0..i]`.
3. **Tối đa hóa tổng độ dài** thay vì tối thiểu hóa: đổi `min` thành `max` trong `best`.
4. **Trả về chính hai đoạn** (chỉ số) thay vì tổng độ dài: lưu thêm chỉ số trong `best` (bài tập 4).
5. **689 — Maximum Sum of 3 Non-Overlapping Subarrays** (độ dài cố định `k`): prefix best từ trái, suffix best từ phải, duyệt đoạn giữa.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Liệt kê đoạn, thử mọi cặp | `O(n²)` | `O(n)` | Kiểm chứng, tìm phản ví dụ |
| 2. Prefix/suffix | `left[i] + right[i+1]` | `O(n)` | `O(n)` | Gợi ý chính thức, dễ chứng minh |
| 3. Prefix + dict + best | Tìm đoạn bằng hash, ghép `best[l−1]` | `O(n)` tb | `O(n)` | Mở rộng được khi có số âm |
| 4. Cửa sổ trượt + best | Hai con trỏ (số dương), một lần quét | `O(n)` | `O(n)` | Khuyên dùng cho đề này |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minSumOfLengths([1,1,1,2,2,2,4,4], 6)` = ?

**Bài 2 (tính tay):** Với `arr = [2,1,3,3,2,3,1]`, `target = 6`, liệt kê mọi đoạn hợp lệ và mảng `best`.

**Bài 3 (điền TODO):** Viết `count_target_subarrays(arr, target)` đếm số đoạn con tổng `target` (phần tử dương) bằng cửa sổ trượt.

**Bài 4 (tự viết lại):** Viết `two_segments(arr, target)` trả về **cặp chỉ số** `((l1, r1), (l2, r2))` của một nghiệm tối ưu (hoặc `None`).

**Bài 5 (mini challenge):** Bỏ điều kiện dương: `arr[i]` có thể âm hoặc 0. Viết `min_sum_general(arr, target)` đúng trong `O(n)` trung bình (gợi ý: prefix dict lưu vị trí **gần nhất**).

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_target_subarrays(arr: List[int], target: int) -> int:
    # TODO: cửa sổ trượt, đếm số r có s == target
    raise NotImplementedError

# Bài 4 — TODO
def two_segments(arr: List[int], target: int):
    # TODO: best lưu (độ dài, l, r) thay vì chỉ độ dài
    raise NotImplementedError

# Bài 5 — TODO
def min_sum_general(arr: List[int], target: int) -> int:
    # TODO: pos[prefix] = vị trí gần nhất; đoạn ngắn nhất kết thúc tại r
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Liệt kê các đoạn tổng 6: `(1..4)` dài 4, `(3..5)` dài 3, `(5..6)` dài 2. (2) `(3..5)` và `(5..6)` chồng nhau ở chỉ số 5; `(1..4)` và `(3..5)` cũng chồng nhau. (3) Chỉ còn cặp `(1..4) + (5..6)` ⇒ `4 + 2 = 6`.

**Bài 2.** (1) Dùng `all_target_subarrays`. (2) `best[i]` là min độ dài các đoạn kết thúc ≤ `i`. (3) Đối chiếu với `trace_sliding`.

**Bài 3.** (1) Copy vòng cửa sổ. (2) `count += (s == target)`. (3) So với `len(all_target_subarrays(...))`.

**Bài 4.** (1) `best[i]` là tuple `(len, l, r)` hoặc `None`. (2) Khi `ans` cải thiện, lưu cả hai đoạn. (3) Kiểm tra: hai đoạn không chồng lấn, tổng đúng `target`, tổng độ dài = đáp án.

**Bài 5.** (1) Không còn "mỗi `r` một `l`", nhưng đoạn **ngắn nhất** kết thúc tại `r` ứng với `l − 1` = vị trí **gần nhất** có prefix `s − target`. (2) Cập nhật `pos[s] = r` **sau** khi tra (ghi đè ⇒ luôn gần nhất). (3) Brute force để đối chiếu với số âm.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, Bài 2
r1 = min_sum_sliding([1, 1, 1, 2, 2, 2, 4, 4], 6)
print("Bài 1:", r1, "| các đoạn:", all_target_subarrays([1, 1, 1, 2, 2, 2, 4, 4], 6))
assert r1 == min_sum_brute([1, 1, 1, 2, 2, 2, 4, 4], 6) == 6

arr2 = [2, 1, 3, 3, 2, 3, 1]
segs2 = all_target_subarrays(arr2, 6)
best2 = []
cur = INF
for r in range(len(arr2)):
    cur = min([cur] + [b - a + 1 for a, b in segs2 if b == r])
    best2.append(cur)
print("Bài 2: đoạn =", segs2, "| best =", best2)
assert segs2 == [(0, 2), (2, 3), (4, 6)] and best2 == [INF, INF, 3, 2, 2, 2, 2]

Bài 1: 6 | các đoạn: [(1, 4), (3, 5), (5, 6)]
Bài 2: đoạn = [(0, 2), (2, 3), (4, 6)] | best = [inf, inf, 3, 2, 2, 2, 2]


In [30]:
def count_target_subarrays_ans(arr, target):
    l = s = cnt = 0
    for r, x in enumerate(arr):
        s += x
        while s > target:
            s -= arr[l]; l += 1
        cnt += s == target
    return cnt

def two_segments_ans(arr, target):
    n = len(arr)
    best = [None] * n
    ans, pair = INF, None
    l = s = 0
    for r in range(n):
        s += arr[r]
        while s > target:
            s -= arr[l]; l += 1
        cand = (r - l + 1, l, r) if s == target else None
        if cand and l > 0 and best[l - 1] and cand[0] + best[l - 1][0] < ans:
            ans = cand[0] + best[l - 1][0]
            pair = (best[l - 1][1:], (l, r))
        prev = best[r - 1] if r else None
        best[r] = min([x for x in (prev, cand) if x], default=None)
    return pair

rng = random.Random(13)
for _ in range(1000):
    arr = [rng.randint(1, 5) for _ in range(rng.randint(1, 20))]
    target = rng.randint(1, 12)
    assert count_target_subarrays_ans(arr, target) == len(all_target_subarrays(arr, target))
    p = two_segments_ans(arr, target)
    want = min_sum_brute(arr, target)
    if p is None:
        assert want == -1
    else:
        (a1, a2), (b1, b2) = p
        assert a2 < b1 and sum(arr[a1:a2 + 1]) == target == sum(arr[b1:b2 + 1])
        assert (a2 - a1 + 1) + (b2 - b1 + 1) == want
print("Bài 3, 4: OK", two_segments_ans([7, 3, 4, 7], 7))

Bài 3, 4: OK ((0, 0), (3, 3))


In [31]:
def min_sum_general_ans(arr, target):
    n = len(arr)
    pos = {0: -1}
    best = [INF] * n
    ans, s = INF, 0
    for r, x in enumerate(arr):
        s += x
        cur = INF
        j = pos.get(s - target)            # vị trí GẦN NHẤT -> đoạn ngắn nhất kết thúc tại r
        if j is not None:
            cur = r - j
            if j >= 0:
                ans = min(ans, cur + best[j])
        best[r] = min(best[r - 1] if r else INF, cur)
        pos[s] = r                         # ghi đè: luôn giữ vị trí gần nhất
    return -1 if ans == INF else ans

def brute_general(arr, target):
    n = len(arr)
    segs = [(i, j) for i in range(n) for j in range(i, n) if sum(arr[i:j + 1]) == target]
    best = min((b - a + 1 + d - c + 1 for a, b in segs for c, d in segs if b < c), default=INF)
    return -1 if best == INF else best

rng = random.Random(17)
for _ in range(1500):
    arr = [rng.randint(-3, 3) for _ in range(rng.randint(1, 14))]
    target = rng.randint(-4, 4)
    assert min_sum_general_ans(arr, target) == brute_general(arr, target), (arr, target)
print("Bài 5: OK (1500 mảng có số âm/0)", min_sum_general_ans([1, -1, 1, -1, 0], 0))

Bài 5: OK (1500 mảng có số âm/0) 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hai đoạn không chồng lấn, ngắn nhất":**

```python
INF = float("inf")
best = [INF] * n                      # đoạn ngắn nhất kết thúc tại <= i
ans, l, s = INF, 0, 0
for r in range(n):
    s += arr[r]
    while s > target:                 # số dương -> cửa sổ trượt
        s -= arr[l]; l += 1
    cur = INF
    if s == target:
        cur = r - l + 1
        if l > 0:
            ans = min(ans, cur + best[l - 1])   # ghép với đoạn nằm HẲN bên trái
    best[r] = min(best[r - 1] if r else INF, cur)
return -1 if ans == INF else ans
```

**Checklist:**
- [ ] `best` khởi tạo `INF`, không phải `0`.
- [ ] Ghép với `best[l − 1]`, không phải `best[r]`.
- [ ] Thu hẹp cửa sổ bằng `while`.
- [ ] `best[r]` luôn được gán (kể cả khi không có đoạn kết thúc tại `r`).
- [ ] Trả `-1` khi `ans == INF`.
- [ ] Đã thử: hai đoạn ngắn nhất chồng lấn, đoạn ngắn nhất "chắn" hai đoạn khác, `n = 1`.